# SPEARMAN CODE

In [8]:
# ============================================================
# Predictive Persistence Analysis (league-selectable) NEW CODE
# ============================================================

# 0) Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import matplotlib.ticker as mtick
# define function which plots first-half and second-half win percentage
# for all teams in one season
def plot_season_scatter(grp, season):
    # compute Spearman rank correlation and p_value between first half and second half win percentage
    r_s, p_s = stats.spearmanr(grp['first'], grp['second'])
    plt.figure(figsize=(6,6)) # creates square figure
    plt.scatter(grp['first'], grp['second'], color='blue') # plots one point per team
    # draw y = x line
    xs = np.linspace(0, 1, 100)
    plt.plot(xs, xs, linestyle='--', color='red', label='y = x (perfect persistence)')
    # fix x and y axis to [0,1], label axes, title graph
    plt.xlim(0, 1)
    plt.ylim(0, 1)
    plt.xlabel('First-half Win %')
    plt.ylabel('Second-half Win %')
    plt.title(f'Predictive Persistence {LEAGUE} – {season}\nSpearman r={r_s:.2f}, p={p_s:.3f}')
    plt.legend() # shows label on y = x line
    plt.show()

# define function which takes output of compute_first_second_halves (one row per team-season
# with 'first' and 'second' win percentages) and computes a per-season Spearman correlation.
def season_correlations(first_second_df):
    out = []  # collect one dict per season with Spearman stats
    league_data = ["Bundesliga", "Premier League", "Serie A", "La Liga"]

    for league in league_data:
        df = first_second_df[first_second_df["league"] == league].copy()
        for season, grp in df.groupby('season'):
            # Need variation and enough teams; otherwise correlation is undefined or meaningless
            if grp['first_half_win_pct'].nunique() <= 1 or grp['second_half_win_pct'].nunique() <= 1 or len(grp) < 3:
                continue
            # rank each array
            r_s, p_s = stats.spearmanr(grp['first_half_win_pct'], grp['second_half_win_pct'])
            out.append({
                'league': league,
                'season': season,
                'n_teams': len(grp),
                'spearman_r': r_s,
                'spearman_p': p_s
            })
      # store stats for the season: number of teams,
      # spearman r-val (correlation between first half and second half win %
      # across teams in that season), p-val
    return pd.DataFrame(out).sort_values('season')

def seeded_season_correlations(first_second_df):
    out = []  # collect one dict per season with Spearman stats
    league_data = ["Bundesliga", "Premier League", "Serie A", "La Liga"]
    for league in league_data:
        for seed in range(1,3):
            df = first_second_df[(first_second_df["seed"] == seed) & (first_second_df["league"] == league)].copy()
            for season, grp in df.groupby('season'):
                # Need variation and enough teams; otherwise correlation is undefined or meaningless
                if grp['first_half_win_pct'].nunique() <= 1 or grp['second_half_win_pct'].nunique() <= 1 or len(grp) < 3:
                    continue
                # rank each array
                r_s, p_s = stats.spearmanr(grp['first_half_win_pct'], grp['second_half_win_pct'])
                out.append({
                    'season': season,
                    'n_teams': len(grp),
                    'spearman_r': r_s,
                    'spearman_p': p_s
                })
      # store stats for the season: number of teams,
      # spearman r-val (correlation between first half and second half win %
      # across teams in that season), p-val
    return pd.DataFrame(out).sort_values('season')


# ------------------------------------------------------------
# 4) Run predictive persistence analysis
# ------------------------------------------------------------

# get per team per season summaries
fs_win_skill = pd.read_csv("/Users/joeyli/skillvsluck/output/european_soccer_leagues/correlations/match_data_correlations/actual_correlation_metric.csv")
# give the summaries to the correlation helper
corr_win_skill = season_correlations(fs_win_skill).sort_values(['league', 'season'])


# ------------------------------------------------------------
# 4) Run predictive persistence analysis
# ------------------------------------------------------------

# get per team per season summaries
# fs_win_sim = pd.read_csv("/Users/joeyli/skillvsluck/output/european_soccer_leagues/correlations/pure_luck_result_based/all_leagues_combined_with_seed.csv")
# # give the summaries to the correlation helper
# corr_win_sim = seeded_season_correlations(fs_win_sim).sort_values(['seed','league', 'season'])



In [9]:
corr_win_skill.to_csv("seasonal_correlations_pure_skill.csv", index=False)

**Run the Following code with your preferred data_type to get the CSV with leagues combined**

In [4]:
csv=fs_win_skill.rename(columns={'first_half_goals': 'first_half_win_pct', 'second_half_goals': 'second_half_win_pct'})
csv.to_csv("actual_correlation_metric.csv", index=False)